# Task 8: 서울 공공자전거 이용정보 통계분석 사이트 (Gradio UI)

## 과제 목표
1. 서울열린데이터광장에서 **3개월간의 공공자전거(따릉이) 일별 이용정보 CSV**를 다운로드한다.
2. 3개 CSV 파일을 **병합(merge)**한다.
3. **Gradio UI**로 기본 통계분석 사이트를 만든다 (여러 가지 화면으로).

## 학습 목표
1. 여러 CSV 파일을 `pd.concat()`으로 병합하는 방법을 배운다.
2. 기술통계(describe) + 시각화(matplotlib)를 조합하여 데이터를 분석한다.
3. `gr.Blocks` + `gr.Tab`으로 **다중 탭 UI**를 구성하는 방법을 익힌다.
4. 데모 발표에 적합한 완성도 높은 앱을 만든다.

---

## ⚠️ 데이터 다운로드 안내

아래 링크에서 **3개월 데이터**를 다운로드하여 이 노트북과 같은 폴더에 저장하세요:

🔗 **다운로드 링크**: https://data.seoul.go.kr/dataList/OA-15246/F/1/datasetView.do

필요한 파일 3개:
- `서울특별시 공공자전거 이용정보(일별)_2510.csv`
- `서울특별시 공공자전거 이용정보(일별)_2511.csv`
- `서울특별시 공공자전거 이용정보(일별)_2512.csv`

> 파일을 이 노트북(`.ipynb`)과 **같은 폴더**에 저장해야 합니다.

---
## 1단계: 환경 설정

### 왜 이렇게 많은 라이브러리가 필요한가?

이 프로젝트는 **데이터 수집 → 전처리 → 분석 → 시각화 → UI 배포**까지의
풀 파이프라인을 구현하므로 각 단계에 적합한 도구가 필요합니다:

| 라이브러리 | 역할 | 단계 |
|-----------|------|------|
| `pandas` | 데이터 로딩, 전처리, 통계 분석 | 수집·전처리·분석 |
| `matplotlib` | 차트 생성 (막대, 선, 파이 등) | 시각화 |
| `gradio` | 웹 UI 구성 (탭, 입력, 출력) | 배포 |
| `glob` | 파일 패턴 매칭으로 CSV 자동 탐색 | 수집 |
| `os` | 파일 경로 처리 | 수집 |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import gradio as gr
import glob
import os

# 한글 폰트 설정 (Windows)
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print(f"pandas: {pd.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"gradio: {gr.__version__}")
print("환경 설정 완료!")

---
## 2단계: CSV 파일 로드 및 병합

### `glob.glob()`으로 파일 자동 탐색

`glob`은 **와일드카드 패턴**으로 파일을 검색하는 표준 라이브러리입니다.

```python
glob.glob("*공공자전거*.csv")
```

이 패턴은 "파일명에 '공공자전거'가 포함되고 .csv로 끝나는 모든 파일"을 찾습니다.
파일명을 직접 하드코딩하는 것보다 **유연하고 유지보수가 쉽습니다.**

### `pd.concat()`으로 여러 DataFrame 병합

#### `concat()` vs `merge()` vs `join()` 비교

| 함수 | 용도 | 비유 |
|------|------|------|
| `pd.concat()` | 같은 구조의 DataFrame을 **위아래로 쌓기** | 같은 양식의 보고서를 묶어서 철하기 |
| `pd.merge()` | 공통 키를 기준으로 **좌우로 합치기** | 학생부 + 성적표를 학번으로 매칭 |
| `df.join()` | 인덱스를 기준으로 **좌우로 합치기** | merge와 유사하나 인덱스 기반 |

3개월 CSV 파일은 **모두 같은 컬럼 구조**를 가지고 있으므로 `concat()`이 적합합니다.

### `ignore_index=True`의 의미

각 CSV 파일의 인덱스는 0부터 시작합니다. `concat()` 기본 동작은 원본 인덱스를 유지하므로:
- 파일1: 0, 1, 2, ..., 999
- 파일2: 0, 1, 2, ..., 999  ← 인덱스 중복!

`ignore_index=True`로 설정하면 새로운 연속 인덱스(0, 1, 2, ..., 2999)를 생성합니다.

In [ ]:
# CSV 파일 자동 탐색
csv_files = sorted(glob.glob("*공공자전거*.csv"))

if len(csv_files) == 0:
    print("⚠️ CSV 파일을 찾을 수 없습니다!")
    print("이 노트북과 같은 폴더에 공공자전거 CSV 파일을 넣어주세요.")
    print(f"현재 디렉토리: {os.getcwd()}")
else:
    print(f"발견된 CSV 파일 ({len(csv_files)}개):")
    for f in csv_files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  - {f} ({size_kb:.0f} KB)")

In [ ]:
# 각 CSV 로드 후 병합
dfs = []
for file in csv_files:
    # encoding='cp949': 서울시 데이터는 한글 인코딩이 cp949(EUC-KR)인 경우가 많음
    # encoding='utf-8'이 실패하면 'cp949' 또는 'euc-kr'로 시도
    try:
        df_temp = pd.read_csv(file, encoding='utf-8')
    except UnicodeDecodeError:
        df_temp = pd.read_csv(file, encoding='cp949')
    
    print(f"{file}: {df_temp.shape[0]:,}행 × {df_temp.shape[1]}열")
    dfs.append(df_temp)

# 병합 — ignore_index=True로 인덱스 재생성
df = pd.concat(dfs, ignore_index=True)

# 컬럼명 양쪽 공백 제거 (서울시 데이터에 ' 이용건수' 같은 공백이 포함됨)
df.columns = df.columns.str.strip()

print(f"\n병합 완료!")
print(f"전체 데이터: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"\n컬럼 목록:")
for col in df.columns:
    print(f"  - '{col}' (타입: {df[col].dtype})")

---
## 3단계: 데이터 전처리

### 왜 전처리가 필요한가?

CSV에서 읽은 데이터는 **모든 값이 문자열**일 수 있습니다.
날짜 컬럼은 `datetime`으로, 숫자 컬럼은 `int`/`float`으로 변환해야
정확한 통계 분석과 시각화가 가능합니다.

### `pd.to_datetime()`의 역할

문자열 `"2025-10-01"`을 pandas의 `Timestamp` 객체로 변환합니다.
이렇게 하면:
- **시간 연산** 가능 (날짜 차이 계산)
- **`.dt` 접근자** 사용 가능 (`.dt.month`, `.dt.dayofweek` 등)
- 시계열 그래프에서 **자동 축 정렬**

### `pd.to_numeric(errors='coerce')`

`errors='coerce'`는 변환할 수 없는 값을 **NaN으로 치환**합니다.
예를 들어 숫자 컬럼에 "-" 같은 문자가 섞여 있으면:
- `errors='raise'` (기본): 에러 발생
- `errors='coerce'`: NaN으로 대체하고 계속 진행
- `errors='ignore'`: 변환 안 하고 원본 유지

In [ ]:
# 컬럼명 확인 (실제 데이터에 맞게 동적으로 처리)
print("현재 컬럼:", list(df.columns))
print("\n처음 5행:")
df.head()

In [ ]:
# 날짜 컬럼 자동 감지 및 변환
# 서울 공공자전거 데이터의 날짜 컬럼은 보통 '대여일자'임
date_candidates = [col for col in df.columns if '일자' in col or '날짜' in col or '대여일' in col]

if date_candidates:
    date_col = date_candidates[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    print(f"날짜 컬럼 '{date_col}' 변환 완료")
    
    # 요일 컬럼 추가 (0=월요일, 6=일요일)
    df['요일'] = df[date_col].dt.day_name()
    df['월'] = df[date_col].dt.month
    print("'요일', '월' 컬럼 추가 완료")
else:
    date_col = None
    print("⚠️ 날짜 컬럼을 자동 감지하지 못했습니다.")
    print(f"컬럼 목록: {list(df.columns)}")

# 숫자 컬럼 자동 변환
numeric_candidates = [col for col in df.columns 
                      if '건수' in col or '거리' in col or '시간' in col 
                      or '운동량' in col or '탄소' in col]

for col in numeric_candidates:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"숫자 컬럼 '{col}' 변환 완료")

print(f"\n전처리 후 데이터 타입:")
print(df.dtypes)

---
## 4단계: 분석 함수 정의

### 왜 함수로 분리하는가?

Gradio의 각 탭/버튼은 **Python 함수**와 연결됩니다.
따라서 각 분석 기능을 독립적인 함수로 만들어야 합니다:

1. **데이터 개요 함수** → 탭 1에 연결
2. **기술통계 함수** → 탭 2에 연결
3. **시각화 함수** → 탭 3에 연결
4. **필터링 함수** → 탭 4에 연결

### matplotlib Figure를 Gradio에 전달하는 방법

Gradio의 `gr.Plot()` 컴포넌트는 `matplotlib.figure.Figure` 객체를 직접 받습니다.
따라서 시각화 함수는 `plt.show()` 대신 **Figure 객체를 반환**해야 합니다.

```python
fig, ax = plt.subplots()  # Figure 객체 생성
ax.plot(...)               # 차트 그리기
plt.close(fig)             # 메모리 누수 방지
return fig                 # Gradio에 전달
```

### `plt.close(fig)`의 중요성

matplotlib는 생성된 Figure를 **전역 상태**로 관리합니다.
`close()` 없이 함수를 반복 호출하면 Figure가 계속 쌓여 **메모리 누수(memory leak)**가 발생합니다.
Gradio처럼 반복 호출되는 환경에서는 반드시 `plt.close()`를 호출해야 합니다.

In [ ]:
# =========================================
# 탭 1: 데이터 개요
# =========================================
def get_data_overview():
    """데이터의 기본 정보를 반환합니다."""
    info_text = (
        f"📊 데이터 개요\n"
        f"{'='*40}\n"
        f"전체 행 수: {df.shape[0]:,}행\n"
        f"전체 열 수: {df.shape[1]}열\n"
        f"컬럼 목록: {', '.join(df.columns)}\n"
        f"결측치 총 수: {df.isnull().sum().sum():,}개\n"
    )
    
    if date_col:
        info_text += (
            f"\n📅 기간 정보\n"
            f"{'='*40}\n"
            f"시작일: {df[date_col].min()}\n"
            f"종료일: {df[date_col].max()}\n"
            f"총 일수: {(df[date_col].max() - df[date_col].min()).days + 1}일\n"
        )
    
    return info_text, df.head(20)

# 테스트
overview_text, overview_df = get_data_overview()
print(overview_text)

In [ ]:
# =========================================
# 탭 2: 기술통계
# =========================================
def get_statistics():
    """주요 수치 컬럼의 기술통계를 반환합니다."""
    # 수치형 컬럼만 선택
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    
    if not numeric_cols:
        return "수치형 컬럼이 없습니다.", pd.DataFrame(), pd.DataFrame()
    
    # describe() — 기본 기술통계
    desc = df[numeric_cols].describe().round(2)
    # reset_index로 통계 항목명을 컬럼으로 변환 (Gradio 표시용)
    desc = desc.reset_index().rename(columns={'index': '통계항목'})
    
    # 결측치 현황
    missing = pd.DataFrame({
        '컬럼명': df.columns,
        '결측치 수': df.isnull().sum().values,
        '결측치 비율(%)': (df.isnull().sum().values / len(df) * 100).round(2)
    })
    missing = missing[missing['결측치 수'] > 0]  # 결측치가 있는 컬럼만
    
    summary = (
        f"📈 기술통계 요약\n"
        f"{'='*40}\n"
        f"수치형 컬럼 수: {len(numeric_cols)}개\n"
        f"결측치 있는 컬럼: {len(missing)}개\n"
    )
    
    return summary, desc, missing if len(missing) > 0 else pd.DataFrame({'정보': ['결측치 없음']})

# 테스트
stat_text, stat_desc, stat_missing = get_statistics()
print(stat_text)
stat_desc

In [ ]:
# =========================================
# 탭 3: 시각화
# =========================================
def create_daily_trend():
    """일별 이용건수 추이 차트를 생성합니다."""
    # 수치형 컬럼 중 '건수' 관련 컬럼 찾기
    count_cols = [col for col in df.columns if '건수' in col or '이용' in col]
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    
    target_col = count_cols[0] if count_cols else (numeric_cols[0] if numeric_cols else None)
    
    if not target_col or not date_col:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.text(0.5, 0.5, '날짜 또는 수치 컬럼을 찾을 수 없습니다', 
                ha='center', va='center', fontsize=14)
        plt.close(fig)
        return fig
    
    # 일별 합계 계산
    daily = df.groupby(date_col)[target_col].sum().reset_index()
    daily = daily.sort_values(date_col)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(daily[date_col], daily[target_col], 
            color='#2196F3', linewidth=1.5, alpha=0.8)
    ax.fill_between(daily[date_col], daily[target_col], alpha=0.15, color='#2196F3')
    ax.set_title(f'일별 {target_col} 추이', fontsize=14, fontweight='bold')
    ax.set_xlabel('날짜')
    ax.set_ylabel(target_col)
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.close(fig)
    return fig

def create_weekday_chart():
    """요일별 평균 이용건수 차트를 생성합니다."""
    count_cols = [col for col in df.columns if '건수' in col or '이용' in col]
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    target_col = count_cols[0] if count_cols else (numeric_cols[0] if numeric_cols else None)
    
    if not target_col or '요일' not in df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.text(0.5, 0.5, '요일 또는 수치 컬럼을 찾을 수 없습니다',
                ha='center', va='center', fontsize=14)
        plt.close(fig)
        return fig
    
    # 요일 순서 지정
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_kr = ['월', '화', '수', '목', '금', '토', '일']
    
    weekday_avg = df.groupby('요일')[target_col].mean()
    weekday_avg = weekday_avg.reindex(day_order)
    
    colors = ['#4CAF50'] * 5 + ['#FF9800'] * 2  # 평일=초록, 주말=주황
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(day_kr, weekday_avg.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'요일별 평균 {target_col}', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'평균 {target_col}')
    ax.grid(True, axis='y', alpha=0.3)
    
    # 각 막대 위에 값 표시
    for bar, val in zip(bars, weekday_avg.values):
        if pd.notna(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{val:,.0f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.close(fig)
    return fig

def create_monthly_chart():
    """월별 총 이용건수 차트를 생성합니다."""
    count_cols = [col for col in df.columns if '건수' in col or '이용' in col]
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    target_col = count_cols[0] if count_cols else (numeric_cols[0] if numeric_cols else None)
    
    if not target_col or '월' not in df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.text(0.5, 0.5, '월 또는 수치 컬럼을 찾을 수 없습니다',
                ha='center', va='center', fontsize=14)
        plt.close(fig)
        return fig
    
    monthly = df.groupby('월')[target_col].sum().reset_index()
    month_labels = [f'{int(m)}월' for m in monthly['월']]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(month_labels, monthly[target_col], 
                  color=['#E91E63', '#9C27B0', '#3F51B5'], 
                  edgecolor='white', linewidth=0.5)
    ax.set_title(f'월별 총 {target_col}', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'총 {target_col}')
    ax.grid(True, axis='y', alpha=0.3)
    
    for bar, val in zip(bars, monthly[target_col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.close(fig)
    return fig

# 테스트
print("시각화 함수 정의 완료!")
create_daily_trend()

In [ ]:
# =========================================
# 탭 3 추가: Top N 대여소 차트
# =========================================
def create_top_stations(n=15):
    """이용건수 상위 대여소 차트를 생성합니다."""
    # 대여소 컬럼 감지: '대여소'가 포함되지만 '번호'나 '구분'은 아닌 컬럼
    station_cols = [col for col in df.columns 
                    if '대여소' in col and '번호' not in col and '구분' not in col]
    count_cols = [col for col in df.columns if '건수' in col]
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    
    station_col = station_cols[0] if station_cols else None
    target_col = count_cols[0] if count_cols else (numeric_cols[0] if numeric_cols else None)
    
    if not station_col or not target_col:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, '대여소 또는 수치 컬럼을 찾을 수 없습니다',
                ha='center', va='center', fontsize=14)
        plt.close(fig)
        return fig
    
    n = int(n)
    top = df.groupby(station_col)[target_col].sum().nlargest(n).reset_index()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(range(len(top)), top[target_col], color='#00BCD4', edgecolor='white')
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top[station_col], fontsize=9)
    ax.invert_yaxis()  # 1위가 맨 위
    ax.set_title(f'{target_col} 상위 {n}개 대여소', fontsize=14, fontweight='bold')
    ax.set_xlabel(f'총 {target_col}')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.close(fig)
    return fig

# 테스트
create_top_stations(10)

In [ ]:
# =========================================
# 탭 4: 필터링
# =========================================
def filter_data(month, gender, age_group, keyword):
    """
    월별 + 성별 + 연령대 + 키워드로 데이터를 필터링합니다.
    """
    filtered = df.copy()
    
    # 월 필터링
    if month and month != "전체" and '월' in df.columns:
        try:
            filtered = filtered[filtered['월'] == int(month)]
        except ValueError:
            pass
    
    # 성별 필터링
    if gender and gender != "전체" and '성별' in df.columns:
        filtered = filtered[filtered['성별'] == gender]
    
    # 연령대 필터링
    if age_group and age_group != "전체" and '연령대' in df.columns:
        filtered = filtered[filtered['연령대'] == age_group]
    
    # 키워드 필터링 (대여소명에서 검색)
    station_cols = [col for col in df.columns 
                    if '대여소' in col and '번호' not in col and '구분' not in col]
    if keyword and keyword.strip() and station_cols:
        station_col = station_cols[0]
        filtered = filtered[filtered[station_col].astype(str).str.contains(keyword.strip(), na=False)]
    
    result_text = f"필터링 결과: {len(filtered):,}행 (전체 {len(df):,}행 중)"
    
    # 최대 500행만 표시 (성능)
    display_df = filtered.head(500)
    if len(filtered) > 500:
        result_text += f" — 상위 500행만 표시"
    
    return result_text, display_df

# 테스트
result_text, result_df = filter_data("전체", "전체", "전체", "")
print(result_text)

---
## 5단계: Gradio Blocks 다중 탭 UI 구성

### `gr.Blocks`의 레이아웃 시스템

Gradio Blocks는 **컨테이너 중첩** 방식으로 레이아웃을 구성합니다:

```
gr.Blocks()                  ← 최상위 컨테이너
  ├── gr.Tabs()              ← 탭 그룹
  │   ├── gr.Tab("탭1")     ← 개별 탭
  │   │   ├── gr.Markdown()  ← 텍스트
  │   │   ├── gr.Row()       ← 수평 배치
  │   │   │   ├── gr.Column()  ← 열1
  │   │   │   └── gr.Column()  ← 열2
  │   │   └── gr.Dataframe() ← 표
  │   ├── gr.Tab("탭2")     ← 다음 탭
  │   └── gr.Tab("탭3")
  └── 하단 영역
```

### `with` 문과 컨텍스트 매니저

Gradio는 Python의 `with` 문을 활용하여 **계층 구조를 표현**합니다.
`with gr.Tab("탭 이름"):` 블록 안에 정의된 컴포넌트는 자동으로 해당 탭에 소속됩니다.
이는 HTML의 `<div>` 중첩과 유사한 개념입니다.

### `demo.launch()` 파라미터

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| `share` | `False` | `True`면 공유 가능한 외부 URL 생성 |
| `server_port` | `7860` | 로컬 서버 포트 |
| `inbrowser` | `True` | 실행 시 자동으로 브라우저 열기 |

In [ ]:
# =========================================
# 성별/연령대 분석 함수 추가 (데모 발표용)
# =========================================
def create_gender_chart():
    """성별 이용건수 비교 차트를 생성합니다."""
    count_cols = [col for col in df.columns if '건수' in col]
    target_col = count_cols[0] if count_cols else None
    
    if not target_col or '성별' not in df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.text(0.5, 0.5, '성별 또는 수치 컬럼을 찾을 수 없습니다',
                ha='center', va='center', fontsize=14)
        plt.close(fig)
        return fig
    
    gender_data = df.groupby('성별')[target_col].sum().reset_index()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#2196F3', '#E91E63', '#9E9E9E'][:len(gender_data)]
    bars = ax.bar(gender_data['성별'], gender_data[target_col], color=colors, edgecolor='white')
    ax.set_title(f'성별 총 {target_col}', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'총 {target_col}')
    ax.grid(True, axis='y', alpha=0.3)
    
    for bar, val in zip(bars, gender_data[target_col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.close(fig)
    return fig

def create_age_chart():
    """연령대별 이용건수 비교 차트를 생성합니다."""
    count_cols = [col for col in df.columns if '건수' in col]
    target_col = count_cols[0] if count_cols else None
    
    if not target_col or '연령대' not in df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.text(0.5, 0.5, '연령대 또는 수치 컬럼을 찾을 수 없습니다',
                ha='center', va='center', fontsize=14)
        plt.close(fig)
        return fig
    
    age_data = df.groupby('연령대')[target_col].sum().sort_values(ascending=False).reset_index()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.viridis([i / len(age_data) for i in range(len(age_data))])
    bars = ax.bar(age_data['연령대'].astype(str), age_data[target_col], color=colors, edgecolor='white')
    ax.set_title(f'연령대별 총 {target_col}', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'총 {target_col}')
    ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    
    for bar, val in zip(bars, age_data[target_col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.close(fig)
    return fig

# =========================================
# Gradio 다중 탭 UI
# =========================================

# 필터용 선택지 미리 계산
month_choices = ["전체"] + [str(m) for m in sorted(df['월'].dropna().unique().astype(int))] if '월' in df.columns else ["전체"]
gender_choices = ["전체"] + sorted(df['성별'].dropna().unique().tolist()) if '성별' in df.columns else ["전체"]
age_choices = ["전체"] + sorted(df['연령대'].dropna().astype(str).unique().tolist()) if '연령대' in df.columns else ["전체"]

with gr.Blocks(
    title="서울 공공자전거 통계분석",
    theme=gr.themes.Soft()
) as demo:
    
    gr.Markdown(
        "# 🚲 서울 공공자전거(따릉이) 이용정보 통계분석\n"
        "3개월간의 일별 이용 데이터를 다양한 관점에서 분석합니다."
    )
    
    with gr.Tabs():
        
        # ── 탭 1: 데이터 개요 ──
        with gr.Tab("📋 데이터 개요"):
            gr.Markdown("### 데이터 기본 정보")
            overview_btn = gr.Button("데이터 정보 불러오기", variant="primary")
            overview_text = gr.Textbox(label="기본 정보", lines=10, interactive=False)
            overview_table = gr.Dataframe(label="데이터 미리보기 (상위 20행)")
            
            overview_btn.click(
                fn=get_data_overview,
                inputs=[],
                outputs=[overview_text, overview_table]
            )
        
        # ── 탭 2: 기술통계 ──
        with gr.Tab("📈 기술통계"):
            gr.Markdown("### 수치 데이터 기술통계 및 결측치 현황")
            stat_btn = gr.Button("통계 분석 실행", variant="primary")
            stat_text = gr.Textbox(label="요약", lines=5, interactive=False)
            stat_table = gr.Dataframe(label="기술통계 (describe)")
            missing_table = gr.Dataframe(label="결측치 현황")
            
            stat_btn.click(
                fn=get_statistics,
                inputs=[],
                outputs=[stat_text, stat_table, missing_table]
            )
        
        # ── 탭 3: 시각화 ──
        with gr.Tab("📊 시각화"):
            gr.Markdown("### 다양한 차트로 데이터를 시각화합니다")
            
            with gr.Row():
                daily_btn = gr.Button("일별 추이", variant="primary")
                weekday_btn = gr.Button("요일별 비교", variant="secondary")
                monthly_btn = gr.Button("월별 비교", variant="secondary")
            
            chart_output = gr.Plot(label="차트")
            
            gr.Markdown("---\n### 성별 · 연령대 분석")
            with gr.Row():
                gender_btn = gr.Button("성별 비교", variant="primary")
                age_btn = gr.Button("연령대별 비교", variant="secondary")
            chart_output2 = gr.Plot(label="인구통계 차트")
            
            gr.Markdown("---\n### 이용량 상위 대여소")
            with gr.Row():
                top_n_input = gr.Number(label="표시할 대여소 수", value=15, minimum=5, maximum=30, precision=0)
                top_btn = gr.Button("상위 대여소 보기", variant="primary")
            top_chart = gr.Plot(label="상위 대여소 차트")
            
            daily_btn.click(fn=create_daily_trend, inputs=[], outputs=[chart_output])
            weekday_btn.click(fn=create_weekday_chart, inputs=[], outputs=[chart_output])
            monthly_btn.click(fn=create_monthly_chart, inputs=[], outputs=[chart_output])
            gender_btn.click(fn=create_gender_chart, inputs=[], outputs=[chart_output2])
            age_btn.click(fn=create_age_chart, inputs=[], outputs=[chart_output2])
            top_btn.click(fn=create_top_stations, inputs=[top_n_input], outputs=[top_chart])
        
        # ── 탭 4: 필터링 ──
        with gr.Tab("🔍 데이터 필터링"):
            gr.Markdown("### 조건별 데이터 조회")
            
            with gr.Row():
                month_input = gr.Dropdown(label="월 선택", choices=month_choices, value="전체")
                gender_input = gr.Dropdown(label="성별", choices=gender_choices, value="전체")
                age_input = gr.Dropdown(label="연령대", choices=age_choices, value="전체")
            with gr.Row():
                keyword_input = gr.Textbox(
                    label="대여소 검색 (키워드)",
                    placeholder="예: 여의도, 강남, 서울역 ...",
                    scale=3
                )
                filter_btn = gr.Button("필터 적용", variant="primary", scale=1)
            
            filter_result = gr.Textbox(label="필터 결과", interactive=False)
            filter_table = gr.Dataframe(label="필터링된 데이터")
            
            filter_btn.click(
                fn=filter_data,
                inputs=[month_input, gender_input, age_input, keyword_input],
                outputs=[filter_result, filter_table]
            )
    
    gr.Markdown(
        "---\n"
        "*데이터 출처: [서울열린데이터광장](https://data.seoul.go.kr/) — 서울특별시 공공자전거 이용정보(일별)*"
    )

# UI 실행
demo.launch()

---
## 핵심 정리

### 이번 과제에서 배운 것

| 개념 | 핵심 내용 |
|------|----------|
| **`pd.concat()`** | 같은 구조의 DataFrame을 위아래로 쌓기. `ignore_index=True`로 인덱스 재생성 |
| **인코딩 처리** | 서울시 CSV는 cp949/euc-kr 인코딩. try/except로 안전하게 처리 |
| **`pd.to_datetime()`** | 문자열 → Timestamp 변환. `.dt` 접근자로 월/요일 추출 |
| **matplotlib → Gradio** | `fig` 객체를 `gr.Plot()`에 전달. `plt.close(fig)` 필수 |
| **`gr.Blocks` + `gr.Tab`** | 다중 탭 레이아웃 구성. `with` 문으로 계층 구조 표현 |
| **`gr.Row()` / `gr.Column()`** | 수평/수직 배치 컨테이너 |

### 데모 발표 포인트

1. **탭 1**: 데이터 규모와 기간 소개
2. **탭 2**: 주요 통계량과 결측치 현황 설명
3. **탭 3**: 일별 추이 → 계절적 패턴, 요일별 → 주말 이용 패턴, 상위 대여소 → 핫스팟
4. **탭 4**: 실시간 필터링 시연 (특정 지역 검색)